# Major Evaluation — Baseline & Model Comparison

This notebook is the **reproducible orchestration layer** for the Major evaluation pipeline. It intentionally keeps research logic inside `src/evaluation/` rather than duplicating implementation inside the notebook.

## Pipeline

```text
Generated captions + references
            │
            ▼
   Data / schema validation
            │
            ▼
   Automatic metric computation
   BLEU / CIDEr / METEOR / ROUGE-L
            │
            ▼
      Human annotations
            │
            ▼
 Human agreement + semantic taxonomy
            │
            ▼
 Metric ↔ human alignment
            │
            ▼
 Model comparison + error analysis
            │
            ▼
 Reproducible evaluation tables/results
```

The notebook is designed to fail loudly when required inputs or evaluation modules are missing. No synthetic human labels are created here.

## 0. Evaluation contract

### Inputs
- `data/annotations/captions_all_models (1).csv` — generated captions and ground-truth references.
- `data/annotation/manual_labels.csv` — real human annotations.

### Reusable implementation
- `src/evaluation/data_validation.py`
- `src/evaluation/metrics.py`
- `src/evaluation/human_alignment.py`
- `src/evaluation/error_analysis.py`
- `src/evaluation/comparison.py`
- `src/evaluation/reporting.py`

### Outputs
All generated evaluation artifacts are written under `results/evaluation/`.

### Canonical human error taxonomy
1. `correct`
2. `attribute_mismatch`
3. `relation_mismatch`
4. `object_misidentification`
5. `object_hallucination`

Human annotations are treated as real experimental data. This notebook does **not** silently substitute simulated labels.

In [ ]:
from pathlib import Path
import sys
import json
import warnings

import pandas as pd
import numpy as np

warnings.filterwarnings('default')

# Resolve repository root from the notebook location when possible.
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'src').exists() and (REPO_ROOT.parent / 'src').exists():
    REPO_ROOT = REPO_ROOT.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

DATA_DIR = REPO_ROOT / 'data'
ANNOTATION_DIR = DATA_DIR / 'annotation'
ANNOTATIONS_DIR = DATA_DIR / 'annotations'
RESULTS_DIR = REPO_ROOT / 'results' / 'evaluation'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

CAPTIONS_PATH = ANNOTATIONS_DIR / 'captions_all_models (1).csv'
MANUAL_LABELS_PATH = ANNOTATION_DIR / 'manual_labels.csv'

print('Repository:', REPO_ROOT)
print('Captions:', CAPTIONS_PATH)
print('Human labels:', MANUAL_LABELS_PATH)
print('Results:', RESULTS_DIR)

In [ ]:
# Required project modules. Fail explicitly if the reusable evaluation package is not ready.
from src.evaluation import data_validation
from src.evaluation import metrics
from src.evaluation import human_alignment
from src.evaluation import error_analysis
from src.evaluation import comparison
from src.evaluation import reporting

print('src.evaluation modules imported successfully.')

## 1. Load and validate evaluation inputs

This stage checks file existence, required columns, image identifiers, model identifiers, duplicate rows, reference availability, and annotation coverage before any metric is computed.

In [ ]:
if not CAPTIONS_PATH.exists():
    raise FileNotFoundError(f'Missing generated-caption file: {CAPTIONS_PATH}')

if not MANUAL_LABELS_PATH.exists():
    raise FileNotFoundError(
        f'Missing real human annotation file: {MANUAL_LABELS_PATH}. '
        'The Major evaluation does not fall back to simulated labels.'
    )

captions_df = pd.read_csv(CAPTIONS_PATH)
labels_df = pd.read_csv(MANUAL_LABELS_PATH)

print('Captions shape:', captions_df.shape)
print('Human-label shape:', labels_df.shape)
print('\nCaption columns:')
print(captions_df.columns.tolist())
print('\nAnnotation columns:')
print(labels_df.columns.tolist())

In [ ]:
# The exact function signatures are owned by src/evaluation/data_validation.py.
# The validation result should contain a machine-readable summary and raise on critical issues.
validation = data_validation.validate_evaluation_inputs(
    captions_df=captions_df,
    annotations_df=labels_df,
)

print(json.dumps(validation, indent=2, default=str))

## 2. Configure models and evaluation settings

The model names below match the existing caption-generation pipeline. Keep this list synchronized with the generated-caption schema.

In [ ]:
MODEL_COLUMNS = {
    'blip': 'blip_caption',
    'blip2': 'blip2_caption',
    'ofa': 'ofa_caption',
    'vit_gpt2': 'vit_gpt2_caption',
}

METRICS = ['bleu1', 'bleu2', 'bleu3', 'bleu4', 'cider', 'meteor', 'rouge_l']
BLEU4_THRESHOLD = 0.20  # Exploratory binary threshold; not a calibrated confidence score.

print('Models:', list(MODEL_COLUMNS))
print('Metrics:', METRICS)
print('Exploratory BLEU-4 threshold:', BLEU4_THRESHOLD)

## 3. Compute automatic caption metrics

Metric implementation lives in `src/evaluation/metrics.py`. The notebook only supplies data and configuration.

The pipeline records sentence/image-level metric values and preserves enough information to reproduce model-level summaries.

In [ ]:
metric_df = metrics.compute_all_metrics(
    captions_df=captions_df,
    model_columns=MODEL_COLUMNS,
    output_path=RESULTS_DIR / 'metrics_all_models.csv',
)

print('Metric dataframe shape:', metric_df.shape)
display(metric_df.head())

In [ ]:
metric_summary = metrics.summarize_metrics(
    metric_df=metric_df,
    model_columns=MODEL_COLUMNS,
    output_path=RESULTS_DIR / 'metric_summary.csv',
)

display(metric_summary)

## 4. Normalize and validate human annotations

Human labels are normalized to the canonical five-class taxonomy. The implementation must reject unsupported categories, inconsistent correctness/error labels, duplicate image-model annotations, and missing required coverage.

In [ ]:
normalized_annotations = human_alignment.normalize_annotations(
    labels_df,
    expected_models=list(MODEL_COLUMNS),
)

normalized_annotations.to_csv(
    RESULTS_DIR / 'normalized_human_annotations.csv',
    index=False,
)

display(normalized_annotations.head())
print('Normalized annotation shape:', normalized_annotations.shape)

## 5. Inter-annotator agreement

Compute agreement from the actual annotation data. Report pairwise Cohen's kappa and Krippendorff's alpha where the annotation structure supports them. Missing labels must not be silently treated as disagreement.

In [ ]:
iaa_results = human_alignment.compute_iaa(
    normalized_annotations,
    output_path=RESULTS_DIR / 'iaa_scores.csv',
)

display(iaa_results)

## 6. Human error taxonomy and semantic error analysis

The analysis distinguishes:
- attribute mismatch,
- relation mismatch,
- object misidentification,
- object hallucination.

This distinction is important for the Major because collapsing semantic error types would weaken the interpretation of metric failure.

In [ ]:
error_distribution = error_analysis.compute_error_distribution(
    normalized_annotations,
    output_path=RESULTS_DIR / 'error_type_distribution.csv',
)

display(error_distribution)

In [ ]:
error_examples = error_analysis.extract_qualitative_examples(
    metric_df=metric_df,
    annotations_df=normalized_annotations,
    output_path=RESULTS_DIR / 'qualitative_error_examples.csv',
)

display(error_examples.head(20))

## 7. Metric ↔ human semantic alignment

This is the central evaluation stage: determine whether automatic metrics agree with human semantic correctness.

The BLEU-4 threshold is used only as an exploratory binary decision boundary. It must not be described as a calibrated probability or confidence score.

In [ ]:
alignment_results = human_alignment.compute_metric_human_alignment(
    metric_df=metric_df,
    annotations_df=normalized_annotations,
    metric='bleu4',
    threshold=BLEU4_THRESHOLD,
    output_path=RESULTS_DIR / 'misalignment_summary.csv',
)

display(alignment_results)

In [ ]:
classifier_report = comparison.build_metric_classifier_report(
    metric_df=metric_df,
    annotations_df=normalized_annotations,
    metric='bleu4',
    threshold=BLEU4_THRESHOLD,
    output_path=RESULTS_DIR / 'metric_classifier_report.csv',
)

display(classifier_report)

## 8. Model comparison

Compare models across automatic metrics, human semantic correctness, error-type distribution, and metric-human misalignment. Do not rank models from one metric alone.

In [ ]:
model_comparison = comparison.compare_models(
    metric_df=metric_df,
    annotations_df=normalized_annotations,
    alignment_df=alignment_results,
    output_path=RESULTS_DIR / 'model_comparison.csv',
)

display(model_comparison)

## 9. Reproducible result tables

Generate the tables used by the Major report. The reporting layer should not alter underlying observations; it only aggregates and formats validated results.

In [ ]:
report_tables = reporting.generate_evaluation_tables(
    metric_df=metric_df,
    metric_summary=metric_summary,
    annotations_df=normalized_annotations,
    iaa_results=iaa_results,
    alignment_results=alignment_results,
    error_distribution=error_distribution,
    model_comparison=model_comparison,
    output_dir=RESULTS_DIR,
)

print('Generated report tables:')
for path in report_tables:
    print(' -', path)

## 10. Final evaluation integrity checks

Before treating the results as Major findings, verify that:

- the input caption file is the intended version;
- every evaluated model has the expected caption column;
- image identifiers are unique where required;
- references are present and parseable;
- every model has the required human annotation coverage;
- no simulated annotation source was used;
- all five canonical semantic categories are represented consistently;
- metric failures were not silently converted to zero;
- all result tables were generated from the same validated run;
- the BLEU-4 threshold is described as exploratory;
- automatic metric agreement is not presented as equivalent to human semantic correctness.

In [ ]:
final_check = reporting.run_final_integrity_check(
    captions_df=captions_df,
    annotations_df=normalized_annotations,
    metric_df=metric_df,
    results_dir=RESULTS_DIR,
)

print(json.dumps(final_check, indent=2, default=str))

if not final_check.get('passed', False):
    raise RuntimeError('Final evaluation integrity check failed. Do not use these results as final Major findings.')

print('\nMajor evaluation completed successfully.')

## 11. Output inventory

The final cell lists all artifacts produced by the evaluation run so the experiment can be audited and reproduced.

In [ ]:
for path in sorted(RESULTS_DIR.rglob('*')):
    if path.is_file():
        print(path.relative_to(REPO_ROOT))

## Interpretation note

The purpose of this notebook is not simply to obtain higher metric scores. The Major evaluation should demonstrate the relationship — and possible mismatch — between conventional caption metrics and human semantic correctness. Results should therefore be interpreted jointly across automatic scores, human correctness, semantic error categories, inter-annotator agreement, and qualitative examples.